# M5: Grid Capacity — i-DE (Iberdrola Distribution)
## IE Sustainability Datathon March 2026, Iberdrola Challenge

**Objective:** Load, clean, and explore the i-DE (Iberdrola Distribución Eléctrica) consumption capacity dataset. Apply the same `grid_status` classification scheme used in M4 and export a model-ready CSV for use by the network optimisation notebook.

**Data source:** i-DE — Consumption capacity map (Mapa de Capacidad de Consumo)  
→ Official page: https://www.i-de.es/conexion-red-electrica/suministro-electrico/mapa-capacidad-consumo  
→ File location: `data/external/grid_capacity_ide_iberdrola/`  
→ Coverage: Central Spain, Castilla y León, Castilla-La Mancha, Murcia, Navarra, Aragón, País Vasco, La Rioja, parts of Galicia and Extremadura

**Role in pipeline:**  
i-DE covers the distribution zones where Iberdrola is the operator — strategically the most important grid dataset for this challenge. Together with M4 (Endesa) it ensures that proposed charging stations in **File 2** receive a valid `grid_status` from their actual local distributor, not a uniform assumption.

**Depends on:** None (standalone)  
**Output:** `data/interim/m5_ide_grid_capacity.csv`

> **⚠️ Download note:** The i-DE portal sometimes restricts direct file downloads. If the automated download below fails, see **Section 1b** for the manual download fallback procedure.

## 0. Setup: Dependencies & Paths

In [ ]:
# !pip install pandas numpy geopandas matplotlib scipy requests openpyxl -q

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
from scipy.spatial import cKDTree

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
IDE_DIR   = '../data/external/grid_capacity_ide_iberdrola/'
RTIG_PATH = '../data/raw/road_routes_spain/carreteras_RTIG.geojson'
OUT_PATH  = '../data/interim/m5_ide_grid_capacity.csv'

os.makedirs(IDE_DIR, exist_ok=True)
os.makedirs('../data/interim', exist_ok=True)

print('Libraries loaded.')
print(f'i-DE data directory: {IDE_DIR}')

## 1a. Automated Download Attempt

The i-DE portal is obligated by CNMC Circular 1/2024 to publish substation-level capacity data. We attempt to download the file directly. If the portal is unavailable (HTTP 503 is common), proceed to **Section 1b**.

Known direct download URLs (try in order — i-DE updates these periodically):

In [ ]:
IDE_DOWNLOAD_URLS = [
    # Primary — i-DE direct CSV
    'https://www.i-de.es/documents/10180/0/Mapa+Capacidad+Consumo.csv',
    # Secondary — XLSX version
    'https://www.i-de.es/documents/10180/0/Mapa+Capacidad+Consumo.xlsx',
    # Tertiary — CNMC unified capacity platform (aggregates all distributors)
    'https://www.generapp.eu/mapa-de-capacidades/',
]

downloaded = False
for url in IDE_DOWNLOAD_URLS[:2]:  # Only try direct file URLs
    ext  = url.split('.')[-1].lower()
    dest = os.path.join(IDE_DIR, f'ide_capacity.{ext}')
    if os.path.exists(dest):
        print(f'File already exists: {dest}')
        downloaded = True
        break
    try:
        r = requests.get(url, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
        r.raise_for_status()
        with open(dest, 'wb') as f:
            f.write(r.content)
        print(f'Downloaded → {dest}  ({len(r.content)/1024:.0f} KB)')
        downloaded = True
        break
    except Exception as e:
        print(f'Failed ({url[:60]}...): {e}')

if not downloaded:
    print()
    print('⚠️  Automated download failed — see Section 1b for manual instructions.')

## 1b. Manual Download Instructions (if 1a failed)

If the automated download above failed, follow these steps:

**Option A — i-DE direct portal:**
1. Go to: https://www.i-de.es/conexion-red-electrica/suministro-electrico/mapa-capacidad-consumo
2. Scroll to the bottom of the page
3. Click the **PDF / CSV / XLSX** download buttons
4. Save the file to: `data/external/grid_capacity_ide_iberdrola/`

**Option B — CNMC unified capacity platform (recommended fallback):**
1. Go to: https://www.generapp.eu/mapa-de-capacidades/
2. Filter by **Distribuidora = "i-DE"** (or "Iberdrola Distribución")
3. Export the filtered layer as CSV
4. Save to: `data/external/grid_capacity_ide_iberdrola/ide_capacity_cnmc.csv`

**Option C — Minimal synthetic fallback (for development only):**  
If neither source is currently accessible, run the cell below to generate a minimal placeholder dataset from known i-DE substations. **This must be replaced with real data before final submission.**

In [ ]:
# ── Check what files exist in the i-DE directory ──────────────────────────────
existing_files = glob.glob(os.path.join(IDE_DIR, '*'))
print(f'Files in {IDE_DIR}:')
if existing_files:
    for f in existing_files:
        size = os.path.getsize(f) / 1024
        print(f'  {os.path.basename(f)}  ({size:.1f} KB)')
else:
    print('  (no files found)')
    print()
    print('Generating minimal placeholder dataset from known i-DE corridor substations...')
    print('⚠️  REPLACE WITH REAL i-DE DATA BEFORE FINAL SUBMISSION')

    # Known major substations near key interurban corridors in the i-DE zone
    placeholder = pd.DataFrame([
        # (node_id, node_name,        municipality,        province,           lat,     lon,   avail_mw, kv)
        ('IDE_001','S.E. Valladolid N','Valladolid',       'Valladolid',      41.653, -4.724,  42.5, 132),
        ('IDE_002','S.E. Burgos Sur',  'Burgos',           'Burgos',          42.328, -3.707,  18.2, 66),
        ('IDE_003','S.E. Salamanca E', 'Salamanca',        'Salamanca',       40.971, -5.624,   9.8, 66),
        ('IDE_004','S.E. Zaragoza NW', 'Zaragoza',         'Zaragoza',        41.669, -0.887,  55.0, 132),
        ('IDE_005','S.E. Albacete',    'Albacete',         'Albacete',        38.993, -1.849,   6.3, 66),
        ('IDE_006','S.E. Pamplona S',  'Pamplona',         'Navarra',         42.788, -1.635,  31.0, 132),
        ('IDE_007','S.E. Logroño E',   'Logroño',          'La Rioja',        42.466, -2.412,  14.0, 66),
        ('IDE_008','S.E. Vitoria',     'Vitoria-Gasteiz',  'Álava',           42.850, -2.651,  22.7, 132),
        ('IDE_009','S.E. Guadalajara', 'Guadalajara',      'Guadalajara',     40.623, -3.162,   1.1, 66),
        ('IDE_010','S.E. Cuenca N',    'Cuenca',           'Cuenca',          40.083, -2.139,   0.5, 66),
        ('IDE_011','S.E. Palencia',    'Palencia',         'Palencia',        42.005, -4.535,  11.8, 66),
        ('IDE_012','S.E. León SW',     'León',             'León',            42.589, -5.618,   7.2, 66),
        ('IDE_013','S.E. Miranda Ebro','Miranda de Ebro',  'Burgos',          42.681, -2.949,   3.0, 66),
        ('IDE_014','S.E. Tudela',      'Tudela',           'Navarra',         42.063, -1.601,  28.5, 132),
        ('IDE_015','S.E. Soria',       'Soria',            'Soria',           41.765, -2.466,   0.8, 66),
    ], columns=['node_id','node_name','municipality','province','latitude','longitude','available_mw','voltage_kv'])

    out_placeholder = os.path.join(IDE_DIR, 'ide_capacity_placeholder.csv')
    placeholder.to_csv(out_placeholder, index=False)
    print(f'Placeholder saved → {out_placeholder} ({len(placeholder)} rows)')

## 2. Load & Standardise

Load whichever file is available in the i-DE directory. We apply the same column mapping strategy as M4, since i-DE and Endesa use similar (but not identical) field naming conventions.

In [ ]:
# ── Find the most recent file in the i-DE directory ───────────────────────────
candidates = sorted(glob.glob(os.path.join(IDE_DIR, '*.csv'))) + \
             sorted(glob.glob(os.path.join(IDE_DIR, '*.xlsx')))

if not candidates:
    raise FileNotFoundError(
        f'No CSV or XLSX file found in {IDE_DIR}.\n'
        'Complete Section 1b manual download instructions first.'
    )

src_file = candidates[-1]   # most recent alphabetically
print(f'Loading: {src_file}')

if src_file.endswith('.xlsx'):
    df_raw = pd.read_excel(src_file)
else:
    for enc in ['utf-8', 'latin1', 'cp1252']:
        try:
            df_raw = pd.read_csv(src_file, sep=';', encoding=enc, decimal=',', low_memory=False)
            # If sep=';' gives a single-column result, try comma
            if df_raw.shape[1] == 1:
                df_raw = pd.read_csv(src_file, sep=',', encoding=enc, decimal='.', low_memory=False)
            print(f'Loaded with encoding={enc}')
            break
        except Exception as e:
            print(f'  {enc} failed: {e}')

print(f'\nRaw shape  : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'\nColumn names (raw):')
for i, col in enumerate(df_raw.columns):
    print(f'  [{i:02d}] {col}')
print(f'\nFirst 3 rows:')
print(df_raw.head(3).to_string())

In [ ]:
# ── Column mapping: i-DE name → internal name ─────────────────────────────────
# Adjust keys to match the printed column names above if needed.
# i-DE uses similar but slightly different naming to Endesa.
COLUMN_MAP = {
    # Node identifier
    'Código':                       'node_id',      # alt: 'Nudo', 'ID', 'Código nudo'
    'Código nudo':                  'node_id',
    # Substation name
    'Nombre':                       'node_name',    # alt: 'Denominación'
    'Denominación':                 'node_name',
    # Municipality
    'Municipio':                    'municipality',
    # Province
    'Provincia':                    'province',
    # Coordinates
    'Latitud':                      'latitude',     # alt: 'LAT'
    'Longitud':                     'longitude',    # alt: 'LON'
    'LAT':                          'latitude',
    'LON':                          'longitude',
    # Available capacity (MW) — critical field
    'Capacidad disponible (MW)':    'available_mw', # alt: 'Capacidad disponible', 'Cap. disp. (MW)'
    'Capacidad disponible':         'available_mw',
    'Cap. disp. (MW)':              'available_mw',
    # Voltage
    'Tensión (kV)':                 'voltage_kv',   # alt: 'Tensión', 'kV'
    'Tensión':                      'voltage_kv',
    'kV':                           'voltage_kv',
}

cols_present = {k: v for k, v in COLUMN_MAP.items() if k in df_raw.columns}
# De-duplicate: if two source cols map to the same target, keep the first match
seen_targets = set()
cols_present_dedup = {}
for k, v in cols_present.items():
    if v not in seen_targets:
        cols_present_dedup[k] = v
        seen_targets.add(v)

df = df_raw[list(cols_present_dedup.keys())].rename(columns=cols_present_dedup).copy()

print(f'Columns mapped : {len(cols_present_dedup)}')
print(f'Mapped columns : {df.columns.tolist()}')
print(f'Shape          : {df.shape}')

## 3. Clean & Validate

Same cleaning pipeline as M4 for consistency across the two grid datasets.

In [ ]:
n_raw = len(df)

# ── Parse numeric fields ──────────────────────────────────────────────────────
for col in ['available_mw', 'latitude', 'longitude', 'voltage_kv']:
    if col in df.columns:
        df[col] = (
            df[col].astype(str)
            .str.replace(',', '.', regex=False)
            .str.strip()
            .replace({'': np.nan, 'nan': np.nan, '-': np.nan, 'None': np.nan})
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')

# ── Drop rows with no coordinates ─────────────────────────────────────────────
df = df.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
n_no_coords = n_raw - len(df)

# ── Validate Spain bounding box ───────────────────────────────────────────────
mask_spain = (
    df['longitude'].between(-20, 5) &
    df['latitude'].between(27, 45)
)
n_outside = (~mask_spain).sum()
df = df[mask_spain].reset_index(drop=True)

# ── Clip negative capacity ────────────────────────────────────────────────────
if 'available_mw' in df.columns:
    n_negative = (df['available_mw'] < 0).sum()
    df['available_mw'] = df['available_mw'].clip(lower=0)
else:
    n_negative = 0

print('CLEANING SUMMARY')
print('-' * 40)
print(f'Raw rows              : {n_raw:>6,}')
print(f'Dropped (no coords)   : {n_no_coords:>6,}')
print(f'Dropped (outside ESP) : {n_outside:>6,}')
print(f'Negative MW clipped   : {n_negative:>6,}  (set to 0)')
print(f'Final clean rows      : {len(df):>6,}')
print()
if 'available_mw' in df.columns:
    print('Capacity (available_mw) statistics:')
    print(df['available_mw'].describe().round(2).to_string())

## 4. Exploratory Data Analysis

The same four-panel EDA as M4, allowing direct visual comparison between the Endesa and i-DE distribution zones.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle(
    'i-DE (Iberdrola Distribución) — Grid Access Capacity EDA\n'
    '(IE Datathon 2026 / Iberdrola Challenge)',
    fontsize=13, fontweight='bold'
)

# ── 1. Capacity distribution ──────────────────────────────────────────────────
ax = axes[0, 0]
if 'available_mw' in df.columns:
    cap = df['available_mw'].dropna()
    cap_capped = cap.clip(upper=cap.quantile(0.99))
    ax.hist(cap_capped, bins=60, color='royalblue', edgecolor='white', linewidth=0.3)
    ax.axvline(cap.median(), color='darkorange', linestyle='--', linewidth=1.5,
               label=f'Median: {cap.median():.1f} MW')
    ax.axvline(cap.mean(), color='crimson', linestyle=':', linewidth=1.5,
               label=f'Mean: {cap.mean():.1f} MW')
    ax.set_title('Distribution of Available Capacity (MW)')
    ax.set_xlabel('Available Capacity (MW)')
    ax.set_ylabel('Node count')
    ax.legend(fontsize=9)
    ax.text(0.97, 0.95, f'n = {len(cap):,} nodes', transform=ax.transAxes,
            ha='right', va='top', fontsize=8, color='grey')

# ── 2. Top 15 provinces by node count ────────────────────────────────────────
ax = axes[0, 1]
if 'province' in df.columns:
    count_col = 'node_id' if 'node_id' in df.columns else 'latitude'
    prov = df.groupby('province').agg(
        nodes=(count_col, 'count'),
        mean_mw=('available_mw', 'mean') if 'available_mw' in df.columns else (count_col, 'count')
    ).sort_values('nodes', ascending=True).tail(15)
    colors_bar = ['#d73027' if m < 5 else '#fee08b' if m < 20 else '#1a9850'
                  for m in prov['mean_mw']]
    bars = ax.barh(prov.index, prov['nodes'], color=colors_bar)
    ax.set_title('Top 15 Provinces by Node Count (i-DE zone)\n(colour = mean available MW: red<5, yellow<20, green≥20)')
    ax.set_xlabel('Number of nodes')
    for bar, (_, row) in zip(bars, prov.iterrows()):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f"{row['mean_mw']:.0f} MW", va='center', fontsize=7.5)

# ── 3. Capacity by voltage level ──────────────────────────────────────────────
ax = axes[1, 0]
if 'voltage_kv' in df.columns and 'available_mw' in df.columns:
    vlt = df.groupby('voltage_kv')['available_mw'].agg(['median', 'count'])
    vlt = vlt.sort_values('median', ascending=False).head(10)
    ax.bar(vlt.index.astype(str), vlt['median'], color='royalblue', alpha=0.8)
    ax.set_title('Median Available Capacity by Voltage Level (kV)')
    ax.set_xlabel('Voltage level (kV)')
    ax.set_ylabel('Median available capacity (MW)')
    ax.tick_params(axis='x', rotation=45)
    for i, (idx, row) in enumerate(vlt.iterrows()):
        ax.text(i, row['median'] + 0.2, f"n={int(row['count'])}",
                ha='center', fontsize=7.5, color='grey')
else:
    ax.text(0.5, 0.5, 'voltage_kv column not found\nUpdate COLUMN_MAP in Section 2',
            ha='center', va='center', transform=ax.transAxes, color='grey')
    ax.set_title('Capacity by Voltage Level (data not available)')

# ── 4. Geographic map ─────────────────────────────────────────────────────────
ax = axes[1, 1]
if 'available_mw' in df.columns:
    scatter = ax.scatter(
        df['longitude'], df['latitude'],
        c=df['available_mw'].clip(upper=df['available_mw'].quantile(0.95)),
        cmap='RdYlGn', s=10, alpha=0.75
    )
    plt.colorbar(scatter, ax=ax, label='Available capacity (MW)', shrink=0.8)
    ax.set_title('i-DE Grid Nodes — Available Capacity\n(green=high, red=low/congested)')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_xlim(-10, 5)
    ax.set_ylim(27, 45)

plt.tight_layout()
plt.savefig('../data/interim/m5_ide_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA charts saved.')

## 5. Grid Status Classification

We apply **identical thresholds to M4** to ensure consistency across the two distributor datasets. All charging locations in Spain will be classified using the same scale regardless of which distributor covers the area.

| Status | Threshold | Reasoning |
|---|---|---|
| **Sufficient** | ≥ 10 MW | Supports a full 60+ charger hub without reinforcement |
| **Moderate** | 1.5 – 10 MW | Supports a standard 10-charger hub; reinforcement needed for scale-up |
| **Congested** | < 1.5 MW | Below minimum for a viable public charging hub at 150 kW/charger |

In [ ]:
# ── Same thresholds as M4 — do not change independently ──────────────────────
THRESHOLD_SUFFICIENT = 10.0
THRESHOLD_MODERATE   =  1.5

def classify_grid(mw):
    if pd.isna(mw):
        return 'Moderate'
    if mw >= THRESHOLD_SUFFICIENT:
        return 'Sufficient'
    elif mw >= THRESHOLD_MODERATE:
        return 'Moderate'
    else:
        return 'Congested'

if 'available_mw' in df.columns:
    df['grid_status'] = df['available_mw'].apply(classify_grid)
else:
    df['grid_status'] = 'Moderate'
    print('WARNING: available_mw column not found — all nodes defaulted to Moderate.')

status_counts = df['grid_status'].value_counts()
status_pct    = (status_counts / len(df) * 100).round(1)

print('GRID STATUS CLASSIFICATION — i-DE')
print('-' * 45)
print(f'  Threshold Sufficient : ≥ {THRESHOLD_SUFFICIENT} MW  (same as M4/Endesa)')
print(f'  Threshold Moderate   : {THRESHOLD_MODERATE} – {THRESHOLD_SUFFICIENT} MW')
print(f'  Threshold Congested  : < {THRESHOLD_MODERATE} MW')
print()
for status in ['Sufficient', 'Moderate', 'Congested']:
    n   = status_counts.get(status, 0)
    pct = status_pct.get(status, 0.0)
    print(f'  {status:<12} : {n:>5,} nodes  ({pct:.1f}%)')
print(f'  {"TOTAL":<12} : {len(df):>5,} nodes')

## 6. Coverage Comparison: i-DE vs Endesa

Side-by-side map comparing the geographic coverage and capacity profiles of both distributor datasets. This is a key visual for the Analytical Report to justify how the two datasets together cover Spain's interurban corridors.

In [ ]:
endesa_path = '../data/interim/m4_endesa_grid_capacity.csv'

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
STATUS_COLORS = {'Sufficient': '#2ca02c', 'Moderate': '#ff7f0e', 'Congested': '#d62728'}

# ── Left: i-DE ────────────────────────────────────────────────────────────────
ax = axes[0]
if os.path.exists(RTIG_PATH):
    gpd.read_file(RTIG_PATH).plot(ax=ax, color='lightgrey', linewidth=0.4, alpha=0.4)
for status, color in STATUS_COLORS.items():
    sub = df[df['grid_status'] == status]
    ax.scatter(sub['longitude'], sub['latitude'], c=color, s=10, alpha=0.7,
               label=f'{status} ({len(sub):,})')
ax.set_title(f'i-DE Grid Nodes (n={len(df):,})', fontsize=11)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_xlim(-10, 5); ax.set_ylim(27, 45)
ax.legend(title='grid_status', fontsize=8, loc='upper left')

# ── Right: Endesa (if M4 has run) ────────────────────────────────────────────
ax = axes[1]
if os.path.exists(endesa_path):
    df_end = pd.read_csv(endesa_path)
    if os.path.exists(RTIG_PATH):
        gpd.read_file(RTIG_PATH).plot(ax=ax, color='lightgrey', linewidth=0.4, alpha=0.4)
    for status, color in STATUS_COLORS.items():
        sub = df_end[df_end['grid_status'] == status]
        ax.scatter(sub['longitude'], sub['latitude'], c=color, s=10, alpha=0.7,
                   label=f'{status} ({len(sub):,})')
    ax.set_title(f'Endesa Grid Nodes (n={len(df_end):,})', fontsize=11)
    ax.legend(title='grid_status', fontsize=8, loc='upper left')
else:
    ax.text(0.5, 0.5,
            'Endesa data not yet available.\nRun M4 first to enable comparison.',
            ha='center', va='center', transform=ax.transAxes,
            fontsize=10, color='grey')
    ax.set_title('Endesa Grid Nodes (M4 not yet run)')

ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_xlim(-10, 5); ax.set_ylim(27, 45)

fig.suptitle(
    'Grid Capacity Coverage: i-DE vs Endesa\n'
    '(Green=Sufficient ≥10 MW | Orange=Moderate 1.5–10 MW | Red=Congested <1.5 MW)',
    fontsize=11
)
plt.tight_layout()
plt.savefig('../data/interim/m5_distributor_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Comparison map saved.')

## 7. Nearest-Substation Lookup Function

Same KD-tree approach as M4. M6 will call `get_grid_status_ide(lat, lon)` for any proposed charging location that falls within the i-DE distribution zone.

**Combined distributor logic for M6:**  
The optimisation notebook should use geographic zone boundaries (or a combined lookup across both datasets) to automatically route each proposed location to the correct distributor. As a simple heuristic: if both M4 and M5 are loaded, take the nearest substation across *both* datasets and use that distributor's classification.

In [ ]:
coords = df[['latitude', 'longitude']].values
tree   = cKDTree(coords)

def get_grid_status_ide(lat, lon):
    """
    Return grid_status for a coordinate using the nearest i-DE substation.

    Parameters
    ----------
    lat : float  Latitude in decimal degrees (WGS84)
    lon : float  Longitude in decimal degrees (WGS84)

    Returns
    -------
    dict with keys: grid_status, available_mw, nearest_node_id, dist_deg, distributor
    """
    dist, idx = tree.query([lat, lon])
    row = df.iloc[idx]
    return {
        'grid_status'    : row['grid_status'],
        'available_mw'   : row.get('available_mw', np.nan),
        'nearest_node_id': row.get('node_id', idx),
        'dist_deg'       : round(dist, 4),
        'distributor'    : 'i-DE'
    }

# ── Smoke test ────────────────────────────────────────────────────────────────
# A-1 near Burgos (i-DE zone)
test_result = get_grid_status_ide(42.34, -3.70)
print('Smoke test — A-1 near Burgos:')
for k, v in test_result.items():
    print(f'  {k:<20} : {v}')

# ── A-3 near Cuenca (i-DE zone, expected low capacity) ───────────────────────
test_result_2 = get_grid_status_ide(40.08, -2.14)
print()
print('Smoke test — A-3 near Cuenca (expected Congested):')
for k, v in test_result_2.items():
    print(f'  {k:<20} : {v}')

print()
print('get_grid_status_ide() is ready for use by M6.')

## 8. Export & Output Verification

Per evaluation criterion T5, all output file structures must be printed and verified.

In [ ]:
export_cols = [c for c in
    ['node_id', 'node_name', 'municipality', 'province',
     'latitude', 'longitude', 'available_mw', 'voltage_kv',
     'grid_status']
    if c in df.columns
]

df[export_cols].to_csv(OUT_PATH, index=False)
size_kb = os.path.getsize(OUT_PATH) / 1024

df_v = pd.read_csv(OUT_PATH)

print('OUTPUT VERIFICATION — m5_ide_grid_capacity.csv')
print('-' * 60)
print(f'  File    : {OUT_PATH}')
print(f'  Rows    : {len(df_v):,}')
print(f'  Columns : {df_v.columns.tolist()}')
print(f'  Size    : {size_kb:.1f} KB')
print()
print('Column dtypes:')
print(df_v.dtypes.to_string())
print()
print('Sample (first 3 rows):')
print(df_v.head(3).to_string(index=False))
print()
print('grid_status distribution:')
print(df_v['grid_status'].value_counts().to_string())
print()
print('Coordinate ranges:')
print(f'  Latitude  : {df_v["latitude"].min():.3f} → {df_v["latitude"].max():.3f}')
print(f'  Longitude : {df_v["longitude"].min():.3f} → {df_v["longitude"].max():.3f}')

# ── Assertions ────────────────────────────────────────────────────────────────
assert len(df_v) > 0, 'ERROR: Output file is empty'
assert 'grid_status' in df_v.columns, 'ERROR: grid_status column missing'
assert set(df_v['grid_status'].unique()).issubset({'Sufficient', 'Moderate', 'Congested'}), \
    'ERROR: Invalid grid_status values'
assert df_v['latitude'].between(27, 45).all(), 'ERROR: Latitude out of Spain range'
assert df_v['longitude'].between(-20, 5).all(), 'ERROR: Longitude out of Spain range'

print()
print('✓ All assertions passed.')

## M5 Summary — Key Takeaways for the Analytical Report

| Finding | Value | Implication |
|---|---|---|
| i-DE distribution zone | Central Spain, Castilla y León, Castilla-La Mancha, Murcia, Navarra, Aragón, País Vasco, La Rioja | Covers A-1, A-2, A-6, N-630 and major radial corridors from Madrid |
| Classification thresholds | Identical to M4 (Sufficient ≥10 MW \| Moderate 1.5–10 MW \| Congested <1.5 MW) | Ensures comparability across both distributors in M6 |
| Lookup method | Nearest-neighbour KD-tree on WGS84 coords | Applied in M6; documented here per Rule 1 spatial matching requirement |
| Strategic note | i-DE is Iberdrola's own distribution arm — congested nodes in this zone are infrastructure priorities for the sponsor | Directly actionable for Iberdrola's deployment roadmap |
| Mandatory rule compliance | Rule 1: grid_status from distributor capacity data ✓ | Thresholds consistent with M4 and documented in Analytical Report |